# Marvedge Task-00046 — V4 Production Pipeline
**FFmpeg Pipe · YuNet CPU · Zero Seeks · FFmpeg-Native Crop**

No GPU required. Target: 22-min 11GB video in under 10 minutes.


In [ ]:
# CELL 1: System setup
import subprocess, os
os.system('apt-get install -qq ffmpeg 2>/dev/null')
os.system('pip install -q scenedetect[opencv] scipy tqdm opencv-python-headless')
# Verify YuNet is available via OpenCV
import cv2
assert hasattr(cv2, 'FaceDetectorYN'), 'Need opencv-python >= 4.5.4 for YuNet'
print(f'OpenCV {cv2.__version__} — YuNet available')
print('All dependencies ready')

In [ ]:
# CELL 2: Clone / pull repo (V4 pipeline is on this branch)
import os
BRANCH = 'feat/task-43-center-crop-fallback'
if not os.path.exists('/content/marvedge'):
    os.system(f'git clone -b {BRANCH} --depth 1 https://github.com/Marvedge/marvedge.git /content/marvedge')
else:
    os.system('git -C /content/marvedge pull')
os.chdir('/content/marvedge')
assert os.path.exists('scripts/ml/benchmark_preprocessing_v4.py'), 'V4 not found'
print('Repo ready — V4 pipeline confirmed')

In [ ]:
# CELL 3: Mount Google Drive + confirm video file
from google.colab import drive
import os, subprocess, shutil
drive.mount('/content/drive', force_remount=True)
print('Video files in MyDrive:')
for f in sorted(os.listdir('/content/drive/MyDrive/')):
    if any(f.lower().endswith(e) for e in ['.mp4','.mkv','.mov','.avi']):
        sz = os.path.getsize(f'/content/drive/MyDrive/{f}') / 1e9
        print(f'  {f}  ({sz:.2f} GB)')

In [ ]:
# CELL 4: Copy Video to Local Colab SSD (CRITICAL FOR SPEED)
import os, shutil
VIDEO_FILENAME = 'kapil.mp4'   # <-- update to match what Cell 3 printed
DRIVE_PATH = f'/content/drive/MyDrive/{VIDEO_FILENAME}'
LOCAL_PATH = f'/content/{VIDEO_FILENAME}'
assert os.path.lexists(DRIVE_PATH), f'Not found: {DRIVE_PATH}'

if not os.path.exists(LOCAL_PATH) or os.path.getsize(LOCAL_PATH) != os.path.getsize(DRIVE_PATH):
    sz = os.path.getsize(DRIVE_PATH) / 1e9
    print(f'Copying {sz:.2f} GB from Google Drive to local SSD (this takes 1-2 mins but saves HOURS of I/O)...')
    shutil.copy2(DRIVE_PATH, LOCAL_PATH)
    print('Copy complete!')
else:
    print('Video already on local SSD.')

VIDEO_PATH = LOCAL_PATH
os.environ['VIDEO_PATH'] = VIDEO_PATH


In [ ]:
# CELL 5: Run V4 pipeline
!python scripts/ml/benchmark_preprocessing_v4.py \
  --videoPath "$VIDEO_PATH" \
  --savePath /content/marvedge/demo/task46_v4 \
  --extractFps 2 \
  --detScale 0.25 \
  --threads 2

In [ ]:
# CELL 6: Download results JSON
import json
from google.colab import files
REPORT = '/content/marvedge/demo/task46_v4/benchmark_report.json'
with open(REPORT) as f: rpt = json.load(f)
print('STAGE BREAKDOWN:')
for s in rpt['stages']:
    print(f"  {s['stage']:<30} {s['wall_time_sec']:>8.2f}s")
print(f"  {'TOTAL':<30} {rpt['total_wall_time_sec']:>8.2f}s")
out = '/content/task46_v4_results.json'
with open(out,'w') as f: json.dump(rpt, f, indent=2)
files.download(out)
print('Results downloaded')